# 🧠 Fine-Tuning Demo — TechNova HR Document
### Suresh D R | AI Product Developer & Technology Mentor
---
## 🎯 What This Notebook Covers:
| Step | What Happens |
|------|--------------|
| 📦 Step 1 | Install libraries (Python 3.12 compatible) |
| 🔇 Step 2 | Imports + GPU check |
| 🤖 Step 3 | Load **GPT-2 Small** (117M params — tiny, fast, free) |
| ❓ Step 4 | **BEFORE training** — ask HR questions → generic wrong answers |
| 📄 Step 5 | **Part A — Self-Supervised FT** on raw TechNova doc (no labels) |
| ✅ Step 6 | Ask same questions → model knows TechNova now! |
| 🏷️ Step 7 | **Part B — Supervised FT** on labelled Q&A pairs |
| 🎯 Step 8 | Ask again → even sharper, direct answers |
| 🔀 Step 9 | Side-by-side comparison: Base vs Self-Supervised vs Supervised |

---
## 🔑 Key Concepts:
```
Self-Supervised Fine-Tuning:
  → Feed raw document text
  → Model predicts next word at every position
  → No labels written by hand — document is its own label ✅

Supervised Fine-Tuning:
  → Feed (Question, Answer) pairs
  → Model learns exact Q→A patterns
  → Labels required — but we create them from the document ✅
```

---
## 📦 STEP 1 — Install Libraries
> Python 3.12 compatible versions pinned below
> ⚠️ After running — **Runtime → Restart Session** → then run from Step 2

In [1]:
# Python 3.12 compatible — pinned versions
!pip install -q transformers==4.40.0
!pip install -q torch==2.2.1
!pip install -q datasets==2.18.0
!pip install -q accelerate==0.29.3
!pip install -q ipywidgets

print('✅ All libraries installed!')
print('👉 Now go to Runtime → Restart Session → run from Step 2')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 25.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 815.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 45.0 MB/s 

In [3]:
!pip install -q "transformers==4.40.2" "accelerate==0.29.3" "peft==0.10.0" "bitsandbytes==0.43.1" datasets
print('✅ Done! Now go to Runtime → Restart Session → run from Step 2')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 8.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.
✅ Done! Now go to Runtime → Restart Session → run from Step 2


---
## 🔇 STEP 2 — Imports, Warnings, GPU Check
> Make sure: **Runtime → Change runtime type → GPU → T4**

In [1]:
import warnings, logging, os, sys, textwrap, json
warnings.filterwarnings('ignore')
logging.disable(logging.CRITICAL)

import torch
from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer,
    TextDataset, DataCollatorForLanguageModeling,
    Trainer, TrainingArguments
)
from datasets import Dataset
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── GPU check ────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU  : {name}')
    print(f'✅ VRAM : {vram:.1f} GB')
    print(f'   GPT-2 Small needs ~0.5 GB — you have plenty!')
    DEVICE = 'cuda'
else:
    print('⚠️  No GPU found!')
    print('   Go to Runtime → Change runtime type → GPU → T4')
    print('   Training will still work on CPU but takes longer.')
    DEVICE = 'cpu'

print(f'\n✅ Python version: {sys.version.split(" ")[0]}')
print(f'✅ PyTorch version: {torch.__version__}')


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

⚠️  No GPU found!
   Go to Runtime → Change runtime type → GPU → T4
   Training will still work on CPU but takes longer.

✅ Python version: 3.12.13
✅ PyTorch version: 2.2.1+cu121


---
## 🤖 STEP 3 — Load GPT-2 Small (117M Parameters)

We use **GPT-2 Small** — the smallest, fastest model that runs on any GPU or even CPU.

| Model | Parameters | VRAM needed | Download size | Login needed? |
|-------|-----------|-------------|---------------|---------------|
| GPT-2 Small | 117 Million | ~0.5 GB | ~500 MB | ❌ None |
| TinyLlama  | 1.1 Billion | ~2.0 GB | ~2 GB   | ❌ None |
| LLaMA 3 8B | 8 Billion   | ~16 GB  | ~16 GB  | ✅ Required |

GPT-2 Small is perfect for learning fine-tuning — same concepts, just faster.

```
GPT-2 Small architecture:
  Layers           :  12 transformer layers
  Hidden dimension :  768  (not 4096 like LLaMA — much smaller)
  Attention heads  :  12
  Vocab size       :  50,257 tokens
  Total params     :  117,000,000
```

In [2]:
MODEL_NAME = 'gpt2'   # GPT-2 Small — 117M params, no login needed

print('Loading GPT-2 Small tokenizer...')
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
print(f'  Vocab size: {tokenizer.vocab_size:,} tokens')
print('✅ Tokenizer loaded')

print('\nLoading GPT-2 Small model...')
base_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
_ = base_model.to(DEVICE)

total_params = sum(p.numel() for p in base_model.parameters())
print(f'\n✅ GPT-2 Small loaded!')
print(f'   Total parameters : {total_params:,}  (~{total_params/1e6:.0f}M)')
print(f'   Model size       : ~{total_params * 4 / 1e6:.0f} MB (float32)')
if DEVICE == 'cuda':
    print(f'   GPU memory used  : {torch.cuda.memory_allocated()/1e9:.2f} GB')

print('\n--- Architecture breakdown ---')
print(f'  Transformer layers  : {base_model.config.n_layer}')
print(f'  Hidden dimension    : {base_model.config.n_embd}')
print(f'  Attention heads     : {base_model.config.n_head}')
print(f'  Context window      : {base_model.config.n_positions} tokens')

# Save original weights for comparison later
import copy
original_state = copy.deepcopy(base_model.state_dict())
print('\n✅ Original weights saved for before/after comparison')

Loading GPT-2 Small tokenizer...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

  Vocab size: 50,257 tokens
✅ Tokenizer loaded

Loading GPT-2 Small model...


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


✅ GPT-2 Small loaded!
   Total parameters : 124,439,808  (~124M)
   Model size       : ~498 MB (float32)

--- Architecture breakdown ---
  Transformer layers  : 12
  Hidden dimension    : 768
  Attention heads     : 12
  Context window      : 1024 tokens

✅ Original weights saved for before/after comparison


---
## ❓ STEP 4 — BEFORE Training: Ask HR Questions

The model has **never seen TechNova's document**.
Pick a question from the dropdown and click **Ask**.

> **Expected result:** Vague, generic, or wrong answers — it knows nothing about TechNova!

Remember the answer. We'll ask the same question after each training stage. 👀

In [3]:
# ── Answer generation helper ─────────────────────────────────────────────────
def generate_answer(mdl, question, max_new_tokens=120):
    prompt = f'Question: {question}\nAnswer:'
    inputs = tokenizer(
        prompt,
        return_tensors = 'pt',
        truncation     = True,
        max_length     = 512
    ).to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens  = max_new_tokens,
            do_sample       = True,
            temperature     = 0.7,
            top_p           = 0.9,
            repetition_penalty = 1.3,
            pad_token_id    = tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    # Clean: stop at double newline
    return raw.split('\n\n')[0].strip() or raw[:300]


QUESTIONS = [
    'What is the maternity leave policy at TechNova?',
    'How many sick leave days do TechNova employees get per year?',
    'What is the ESOP vesting schedule at TechNova?',
    'How is leave encashment calculated at TechNova?',
    'What is the L&D budget for TechNova employees?',
    'What are the salary bands for L3 employees at TechNova?',
    'What is the work from home policy at TechNova?',
    'How is the annual performance review done at TechNova?',
]

style_w = {'description_width': '90px'}

dd0 = widgets.Dropdown(
    options     = QUESTIONS,
    description = '❓ Question:',
    style       = style_w,
    layout      = widgets.Layout(width='90%')
)
btn0 = widgets.Button(
    description  = '🤖  Ask (BASE model)',
    button_style = 'danger',
    layout       = widgets.Layout(width='220px', height='38px')
)
out0 = widgets.Output()

def on0(b):
    with out0:
        clear_output(wait=True)
        q = dd0.value
        display(HTML(
            f"<div style='background:#fef2f2;border-left:4px solid #dc2626;"
            f"padding:12px;border-radius:6px;margin:8px 0'>"
            f"<b>❓ {q}</b><br>"
            f"<span style='color:#991b1b'>⏳ Generating...</span></div>"
        ))
        ans = generate_answer(base_model, q)
        display(HTML(
            f"<div style='background:#fff1f2;border-left:4px solid #f43f5e;"
            f"padding:14px;border-radius:6px;font-family:monospace;font-size:13px;line-height:1.8'>"
            f"<b>🤖 BASE model answer:</b><br><br>{ans}</div>"
            f"<div style='background:#fee2e2;padding:8px 12px;border-radius:6px;margin-top:6px;font-size:13px'>"
            f"⚠️ Generic / wrong — no TechNova knowledge yet!</div>"
        ))

btn0.on_click(on0)
display(HTML('<h4>Select a question → click Ask to see the base model response 👇</h4>'))
display(dd0)
print()
display(btn0)
display(out0)

Dropdown(description='❓ Question:', layout=Layout(width='90%'), options=('What is the maternity leave policy a…

Button(button_style='danger', description='🤖  Ask (BASE model)', layout=Layout(height='38px', width='220px'), …

Output()

---
## 📄 STEP 5 — Part A: Self-Supervised Fine-Tuning
### (No Labels Needed — Raw Document Only)

**How it works — Next Word Prediction:**
```
Document text → chopped into 128-token windows

For every position in the window:
  Input seen so far          →   Next word to predict
  ─────────────────────────────────────────────────────
  'Maternity leave at'       →   'TechNova'
  'Maternity leave at TechNova' →  'is'
  'Maternity leave at TechNova is' → '26'
  'ESOP vesting begins after' →  '2'
  'ESOP vesting begins after 2' → 'years'

No human wrote these labels.
The document generates them automatically. ✅
One document → hundreds of training examples.
```

**Training parameters explained:**
```
num_train_epochs = 5    → model reads the full document 5 times
learning_rate    = 5e-5 → step size for weight updates (0.00005)
batch_size       = 4    → 4 chunks processed before each weight update
```

In [4]:
import os, copy

# ── TechNova HR Document (built-in) ──────────────────────────────────────────
TECHNOVA_DOC = """TechNova HR Policy Document

LEAVE POLICY
Maternity Leave: TechNova provides 26 weeks of fully paid maternity leave.
Employees must have completed 80 days of continuous service to be eligible.
Applications must be submitted at least 30 days in advance through HRConnect.
Maternity leave salary is paid via standard payroll cycle, not in advance.

Paternity Leave: TechNova provides 5 days of paid paternity leave.
Must be taken within 30 days of the child birth.

Sick Leave: Employees receive 10 sick leave days per calendar year.
Sick leave cannot be carried forward to the next year.
For leaves exceeding 3 consecutive days a medical certificate is required.

Earned Leave: Employees accrue 1.5 days of earned leave per month which is 18 days per year.
A maximum of 30 days of earned leave can be carried forward to the next year.

Leave Encashment: Employees can encash up to 30 days of earned leave per year.
The encashment rate is calculated as Basic Salary divided by 26 working days.
Applications must be submitted through HRConnect by the 10th of each month.
Leave encashment applies to earned leave only, not to maternity or paternity leave.

SALARY AND COMPENSATION
L1 employees earn Rs 40000 to Rs 70000 per month.
L2 employees earn Rs 70000 to Rs 110000 per month.
L3 employees earn Rs 140000 to Rs 210000 per month.
L4 employees earn Rs 250000 to Rs 400000 per month.
Salary is credited on the last working day of each month via NEFT transfer.
Performance bonus is paid annually in April based on the annual appraisal result.

ESOP POLICY
ESOP stands for Employee Stock Option Plan.
ESOP vesting at TechNova begins after 2 years of continuous service.
Employees at L3 grade and above are eligible for ESOP.
25 percent of options vest after year 1 of the vesting period.
Then 25 percent vest each subsequent year over 4 years total.
The grant price is fixed at the time of joining and does not change.
ESOP options must be exercised within 90 days of leaving the company.

PERFORMANCE REVIEW
TechNova conducts annual performance reviews every March.
Performance is rated on a scale of 1 to 5.
A rating of 4 or above qualifies for a fast-track promotion consideration.
Annual increment ranges from 8 percent to 25 percent based on performance rating.
Promotion decisions are communicated by end of April each year.

LEARNING AND DEVELOPMENT
Each employee receives an L&D budget of Rs 50000 per year.
The budget can be used for online courses, certifications, conferences and books.
Unused L&D budget does not carry forward to the next year.
Applications must be approved by the reporting manager before spending.

WORK FROM HOME POLICY
TechNova follows a hybrid model: 3 days in office and 2 days work from home per week.
Monday and Friday are mandatory office days for all employees.
Work from home is not applicable during the probation period of 6 months.

CONTACT
For all HR queries contact hr@technova.com or raise a ticket on HRConnect portal.
HR helpdesk is available Monday to Friday 9 AM to 6 PM IST."""

# Save document to file (required by TextDataset)
with open('technova_hr.txt', 'w') as f:
    f.write(TECHNOVA_DOC)

print('=' * 55)
print('PART A — SELF-SUPERVISED FINE-TUNING')
print('=' * 55)
print(f'Document : technova_hr.txt')
print(f'Words    : {len(TECHNOVA_DOC.split())}')
print(f'Labels   : NONE — model predicts next word automatically')
print()

# ── Tokenise and chunk the document ──────────────────────────────────────────
print('Step 1/4 — Tokenising document...')
tokens = tokenizer(
    TECHNOVA_DOC,
    return_tensors       = 'pt',
    truncation           = False,
    add_special_tokens   = False
)['input_ids'][0]
print(f'  Document tokens   : {len(tokens)}')

CHUNK = 128   # each training example is 128 tokens
stride = 64   # 50% overlap so no info is missed at boundaries
chunks = []
for start in range(0, len(tokens) - CHUNK + 1, stride):
    c = tokens[start: start + CHUNK].tolist()
    chunks.append({'input_ids': c, 'labels': c})  # label = input (next-token pred)

self_sup_ds = Dataset.from_list(chunks)
print(f'  Training chunks   : {len(self_sup_ds)}')
print(f'  Each chunk        : {CHUNK} tokens')
print(f'  Label source      : document itself (next-token prediction)')
print()

# ── Copy the base model — we fine-tune the copy ───────────────────────────────
print('Step 2/4 — Preparing model copy for self-supervised training...')
self_sup_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
_ = self_sup_model.to(DEVICE)
print('  Model copy ready  ✅')
print()

# ── Training arguments ────────────────────────────────────────────────────────
print('Step 3/4 — Training configuration:')
print('  num_train_epochs = 5      → 5 full passes over the document')
print('  learning_rate    = 5e-5   → 0.00005 — careful small steps')
print('  batch_size       = 4      → 4 chunks per weight update')
print('  warmup_steps     = 10     → slowly ramp up LR at start')
print()

ss_args = TrainingArguments(
    output_dir                  = './ss_output',
    num_train_epochs            = 5,
    per_device_train_batch_size = 4,
    learning_rate               = 5e-5,
    warmup_steps                = 10,
    weight_decay                = 0.01,
    logging_steps               = 2,
    save_strategy               = 'no',
    report_to                   = 'none',
    no_cuda                     = (DEVICE == 'cpu'),
    fp16                        = (DEVICE == 'cuda'),
)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

ss_trainer = Trainer(
    model         = self_sup_model,
    args          = ss_args,
    train_dataset = self_sup_ds,
    data_collator = collator,
)

print('Step 4/4 — Training started!')
print('  Watch the loss ↓ — it should decrease each step')
print('  Lower loss = model getting better at predicting TechNova text')
print('-' * 55)

ss_trainer.train()

print('-' * 55)
print('✅ Self-supervised fine-tuning complete!')
print('   Model has now read TechNova HR document!')
print('   Move to Step 6 to test it → 🎉')

PART A — SELF-SUPERVISED FINE-TUNING
Document : technova_hr.txt
Words    : 511
Labels   : NONE — model predicts next word automatically

Step 1/4 — Tokenising document...
  Document tokens   : 719
  Training chunks   : 10
  Each chunk        : 128 tokens
  Label source      : document itself (next-token prediction)

Step 2/4 — Preparing model copy for self-supervised training...
  Model copy ready  ✅

Step 3/4 — Training configuration:
  num_train_epochs = 5      → 5 full passes over the document
  learning_rate    = 5e-5   → 0.00005 — careful small steps
  batch_size       = 4      → 4 chunks per weight update
  warmup_steps     = 10     → slowly ramp up LR at start

Step 4/4 — Training started!
  Watch the loss ↓ — it should decrease each step
  Lower loss = model getting better at predicting TechNova text
-------------------------------------------------------


Step,Training Loss
2,3.790400
4,3.656700
6,3.385600
8,3.284100
10,3.163900
12,2.664300
14,2.541000


-------------------------------------------------------
✅ Self-supervised fine-tuning complete!
   Model has now read TechNova HR document!
   Move to Step 6 to test it → 🎉


---
## ✅ STEP 6 — After Self-Supervised Training: Ask Again!

Pick the **same question** you asked in Step 4.

> **Expected:** Model now gives **TechNova-specific answers** — actual numbers like 26 weeks, Rs 50,000, 2 years, etc.

The model learned these facts **with zero labelling** — just by reading the raw document.

In [5]:
style_w = {'description_width': '90px'}

dd1 = widgets.Dropdown(
    options     = QUESTIONS,
    description = '❓ Question:',
    style       = style_w,
    layout      = widgets.Layout(width='90%')
)
btn1 = widgets.Button(
    description  = '✅  Ask (Self-Supervised)',
    button_style = 'warning',
    layout       = widgets.Layout(width='240px', height='38px')
)
out1 = widgets.Output()

def on1(b):
    with out1:
        clear_output(wait=True)
        q = dd1.value
        display(HTML(
            f"<div style='background:#fefce8;border-left:4px solid #ca8a04;"
            f"padding:12px;border-radius:6px;margin:8px 0'>"
            f"<b>❓ {q}</b><br>"
            f"<span style='color:#92400e'>⏳ Generating...</span></div>"
        ))
        ans = generate_answer(self_sup_model, q)
        display(HTML(
            f"<div style='background:#fffbeb;border-left:4px solid #f59e0b;"
            f"padding:14px;border-radius:6px;font-family:monospace;font-size:13px;line-height:1.8'>"
            f"<b>🧠 After Self-Supervised FT:</b><br><br>{ans}</div>"
            f"<div style='background:#fef9c3;padding:8px 12px;border-radius:6px;margin-top:6px;font-size:13px'>"
            f"📘 Trained on: raw TechNova doc — zero labels! ✅</div>"
        ))

btn1.on_click(on1)
display(HTML('<h4>Ask the same question as Step 4 → see the improvement! 👇</h4>'))
display(dd1)
print()
display(btn1)
display(out1)

Dropdown(description='❓ Question:', layout=Layout(width='90%'), options=('What is the maternity leave policy a…

Button(button_style='warning', description='✅  Ask (Self-Supervised)', layout=Layout(height='38px', width='240…

Output()

---
## 🏷️ STEP 7 — Part B: Supervised Fine-Tuning
### (With Labelled Q&A Pairs)

**How supervised fine-tuning is different:**
```
Self-Supervised:              Supervised:
  Input:  any text chunk        Input:  a specific Question
  Label:  next word             Label:  the exact correct Answer

  Result: learns domain vocab   Result: learns exact Q→A patterns
```

**We create 10 labelled Q&A pairs from the TechNova document:**
```
Q: What is the maternity leave policy at TechNova?
A: TechNova provides 26 weeks of fully paid maternity leave...

Q: What is the ESOP vesting schedule?
A: ESOP vesting at TechNova begins after 2 years of service...
```

**In real projects:** generate these with GPT (~$2 for 500 pairs) or write them manually.

**Training parameters:**
```
num_train_epochs = 8    → more epochs because dataset is small (10 pairs)
learning_rate    = 3e-5 → slightly lower than self-supervised — fine details
batch_size       = 2    → smaller batch — small dataset
```

In [ ]:
QA_PAIRS = [
    {
        'question': 'What is the maternity leave policy at TechNova?',
        'answer'  : 'TechNova provides 26 weeks of fully paid maternity leave. Employees must have completed 80 days of continuous service to be eligible. Applications must be submitted at least 30 days in advance through HRConnect.'
    },
    {
        'question': 'How many sick leave days do TechNova employees get per year?',
        'answer'  : 'TechNova employees receive 10 sick leave days per calendar year. Sick leave cannot be carried forward to the next year. For leaves exceeding 3 consecutive days, a medical certificate is required.'
    },
    {
        'question': 'What is the ESOP vesting schedule at TechNova?',
        'answer'  : 'ESOP vesting at TechNova begins after 2 years of continuous service. Employees at L3 grade and above are eligible. 25 percent of options vest after year 1, then 25 percent each year over 4 years total.'
    },
    {
        'question': 'How is leave encashment calculated at TechNova?',
        'answer'  : 'At TechNova, leave encashment rate is Basic Salary divided by 26 working days. Employees can encash up to 30 days of earned leave per year. Applications must be submitted through HRConnect by the 10th of each month.'
    },
    {
        'question': 'What is the L&D budget for TechNova employees?',
        'answer'  : 'Each TechNova employee receives an L&D budget of Rs 50000 per year. The budget can be used for online courses, certifications, conferences and books. Unused L&D budget does not carry forward to the next year.'
    },
    {
        'question': 'What are the salary bands for L3 employees at TechNova?',
        'answer'  : 'L3 employees at TechNova earn between Rs 140000 and Rs 210000 per month. Salary is credited on the last working day of each month via NEFT transfer.'
    },
    {
        'question': 'What is the work from home policy at TechNova?',
        'answer'  : 'TechNova follows a hybrid model with 3 days in office and 2 days work from home per week. Monday and Friday are mandatory office days. Work from home is not applicable during the probation period of 6 months.'
    },
    {
        'question': 'How is the annual performance review done at TechNova?',
        'answer'  : 'TechNova conducts annual performance reviews every March. Performance is rated on a scale of 1 to 5. A rating of 4 or above qualifies for fast-track promotion. Annual increment ranges from 8 to 25 percent based on rating.'
    },
    {
        'question': 'What are the salary bands for L1 and L2 employees?',
        'answer'  : 'L1 employees at TechNova earn Rs 40000 to Rs 70000 per month. L2 employees earn Rs 70000 to Rs 110000 per month.'
    },
    {
        'question': 'What is the paternity leave policy at TechNova?',
        'answer'  : 'TechNova provides 5 days of paid paternity leave. This must be taken within 30 days of the child birth.'
    },
]

print('=' * 55)
print('PART B — SUPERVISED FINE-TUNING')
print('=' * 55)
print(f'Labelled Q&A pairs : {len(QA_PAIRS)}')
print(f'Labels             : YES — exact question + correct answer')
print()
print('Training examples:')
for i, qa in enumerate(QA_PAIRS[:3]):
    print(f'  [{i+1}] Q: {qa["question"][:55]}...')
    print(f'       A: {qa["answer"][:55]}...')
print(f'  ... and {len(QA_PAIRS)-3} more pairs')
print()

# ── Format as training text ───────────────────────────────────────────────────
print('Step 1/4 — Tokenising Q&A pairs...')

def format_qa(qa):
    return f"Question: {qa['question']}\nAnswer: {qa['answer']}{tokenizer.eos_token}"

formatted = [format_qa(qa) for qa in QA_PAIRS]
formatted  = formatted * 8   # 10 pairs × 8 = 80 training examples

def tokenize_fn(examples):
    enc = tokenizer(
        examples['text'],
        truncation = True,
        max_length = 256,
        padding    = 'max_length'
    )
    enc['labels'] = enc['input_ids'][:]   # ← slice instead of .copy() — fixes NumPy 2.x error
    return enc

raw_sup_ds = Dataset.from_dict({'text': formatted})
sup_ds     = raw_sup_ds.map(tokenize_fn, batched=True, remove_columns=['text'])
# NOTE: set_format(type='torch') removed — causes NumPy 2.x ValueError

print(f'  Original Q&A pairs: {len(QA_PAIRS)}')
print(f'  After repetition  : {len(formatted)} (×8 repeats for small dataset)')
print(f'  Tokenised examples: {len(sup_ds)}')
print()

# ── Fresh model copy for supervised training ──────────────────────────────────
print('Step 2/4 — Preparing fresh model copy for supervised training...')
sup_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
_ = sup_model.to(DEVICE)
print('  Model copy ready  ✅')
print()

print('Step 3/4 — Training configuration:')
print('  num_train_epochs = 8      → more epochs — small labelled dataset')
print('  learning_rate    = 3e-5   → 0.00003 — fine-grained learning')
print('  batch_size       = 2      → smaller batch for small dataset')
print()

sup_args = TrainingArguments(
    output_dir                  = './sup_output',
    num_train_epochs            = 8,
    per_device_train_batch_size = 2,
    learning_rate               = 3e-5,
    warmup_steps                = 20,
    weight_decay                = 0.01,
    logging_steps               = 5,
    save_strategy               = 'no',
    report_to                   = 'none',
    no_cuda                     = (DEVICE == 'cpu'),
    fp16                        = (DEVICE == 'cuda'),
)

sup_trainer = Trainer(
    model         = sup_model,
    args          = sup_args,
    train_dataset = sup_ds,
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print('Step 4/4 — Training started!')
print('  Each example: Question → Answer (exact labels)')
print('  Loss = how wrong the predicted answer is vs correct answer')
print('-' * 55)

sup_trainer.train()

print('-' * 55)
print('✅ Supervised fine-tuning complete!')
print('   Model now knows exact Q→A patterns from TechNova!')
print('   Move to next step to test it → 🎉')

PART B — SUPERVISED FINE-TUNING
Labelled Q&A pairs : 10
Labels             : YES — exact question + correct answer

Training examples:
  [1] Q: What is the maternity leave policy at TechNova?...
       A: TechNova provides 26 weeks of fully paid maternity leav...
  [2] Q: How many sick leave days do TechNova employees get per ...
       A: TechNova employees receive 10 sick leave days per calen...
  [3] Q: What is the ESOP vesting schedule at TechNova?...
       A: ESOP vesting at TechNova begins after 2 years of contin...
  ... and 7 more pairs

Step 1/4 — Tokenising Q&A pairs...


Map:   0%|          | 0/80 [00:00<?, ? examples/s]

  Original Q&A pairs: 10
  After repetition  : 80 (×8 repeats for small dataset)
  Tokenised examples: 80

Step 2/4 — Preparing fresh model copy for supervised training...
  Model copy ready  ✅

Step 3/4 — Training configuration:
  num_train_epochs = 8      → more epochs — small labelled dataset
  learning_rate    = 3e-5   → 0.00003 — fine-grained learning
  batch_size       = 2      → smaller batch for small dataset

Step 4/4 — Training started!
  Each example: Question → Answer (exact labels)
  Loss = how wrong the predicted answer is vs correct answer
-------------------------------------------------------


Step,Training Loss
5,3.540000
10,3.440100
15,3.166300
20,2.798000
25,2.215100
30,2.178200
35,1.772800
40,1.512600
45,1.256000
50,1.111800


Step,Training Loss
5,3.540000
10,3.440100
15,3.166300
20,2.798000
25,2.215100
30,2.178200
35,1.772800
40,1.512600
45,1.256000
50,1.111800


---
## 🎯 STEP 8 — After Supervised Training: Ask Again!

Pick the same question. This model was trained on **exact Q→A pairs**.

> **Expected:** Even more direct and accurate answers — model knows the exact format and numbers.

In [ ]:
style_w = {'description_width': '90px'}

dd2 = widgets.Dropdown(
    options     = QUESTIONS,
    description = '❓ Question:',
    style       = style_w,
    layout      = widgets.Layout(width='90%')
)
btn2 = widgets.Button(
    description  = '🎯  Ask (Supervised)',
    button_style = 'success',
    layout       = widgets.Layout(width='220px', height='38px')
)
out2 = widgets.Output()

def on2(b):
    with out2:
        clear_output(wait=True)
        q = dd2.value
        display(HTML(
            f"<div style='background:#f0fdf4;border-left:4px solid #16a34a;"
            f"padding:12px;border-radius:6px;margin:8px 0'>"
            f"<b>❓ {q}</b><br>"
            f"<span style='color:#166534'>⏳ Generating...</span></div>"
        ))
        ans = generate_answer(sup_model, q)
        display(HTML(
            f"<div style='background:#f0fdf4;border-left:4px solid #16a34a;"
            f"padding:14px;border-radius:6px;font-family:monospace;font-size:13px;line-height:1.8'>"
            f"<b>🎯 After Supervised FT:</b><br><br>{ans}</div>"
            f"<div style='background:#dcfce7;padding:8px 12px;border-radius:6px;margin-top:6px;font-size:13px'>"
            f"🏷️ Trained on: 10 labelled Q&A pairs ✅</div>"
        ))

btn2.on_click(on2)
display(HTML('<h4>Ask any question → see the supervised model response! 👇</h4>'))
display(dd2)
print()
display(btn2)
display(out2)

---
## 🔀 STEP 9 — Side-by-Side: Base vs Self-Supervised vs Supervised

The best cell in the notebook — see all three answers in one click!

| Model | Trained on | Labels needed? |
|-------|-----------|----------------|
| ❌ Base GPT-2 | General internet text | — |
| 🧠 Self-Supervised | TechNova raw document | ❌ No |
| 🎯 Supervised | TechNova Q&A pairs | ✅ Yes |

In [ ]:
style_w = {'description_width': '90px'}

dd3 = widgets.Dropdown(
    options     = QUESTIONS,
    description = '❓ Question:',
    style       = style_w,
    layout      = widgets.Layout(width='90%')
)
btn3 = widgets.Button(
    description  = '🔀  Compare All 3 Models',
    button_style = 'primary',
    layout       = widgets.Layout(width='240px', height='40px')
)
out3 = widgets.Output()

def on3(b):
    with out3:
        clear_output(wait=True)
        q = dd3.value
        display(HTML(
            f"<div style='background:#f8faff;border-left:5px solid #6366f1;"
            f"padding:12px;border-radius:6px;margin:8px 0'>"
            f"<b>❓ {q}</b><br>"
            f"<span style='color:#4338ca'>⏳ Running all 3 models — takes ~30 seconds...</span></div>"
        ))

        a0 = generate_answer(base_model,     q)
        a1 = generate_answer(self_sup_model, q)
        a2 = generate_answer(sup_model,      q)

        display(HTML(
            f"<table style='width:100%;border-collapse:collapse;font-size:12.5px;margin-top:10px'>"
            f"<tr>"
            f"<th style='background:#fef2f2;padding:10px;border:1px solid #fca5a5;width:33%'>❌ BASE GPT-2<br><span style='font-weight:400;font-size:11px'>No TechNova training</span></th>"
            f"<th style='background:#fefce8;padding:10px;border:1px solid #fde68a;width:33%'>🧠 SELF-SUPERVISED<br><span style='font-weight:400;font-size:11px'>Raw doc, no labels</span></th>"
            f"<th style='background:#f0fdf4;padding:10px;border:1px solid #86efac;width:33%'>🎯 SUPERVISED<br><span style='font-weight:400;font-size:11px'>Labelled Q&A pairs</span></th>"
            f"</tr>"
            f"<tr>"
            f"<td style='padding:12px;border:1px solid #fca5a5;vertical-align:top;font-family:monospace;line-height:1.7'>{a0}</td>"
            f"<td style='padding:12px;border:1px solid #fde68a;vertical-align:top;font-family:monospace;line-height:1.7'>{a1}</td>"
            f"<td style='padding:12px;border:1px solid #86efac;vertical-align:top;font-family:monospace;line-height:1.7'>{a2}</td>"
            f"</tr>"
            f"</table>"
        ))

btn3.on_click(on3)
display(HTML('<h4>Pick any question → compare all 3 models side by side! 👇</h4>'))
display(dd3)
print()
display(btn3)
display(out3)

---
## 🏁 Summary
```
╔══════════════════════════════════════════════════════════════╗
║   🧠 FINE-TUNING DEMO — WHAT YOU LEARNED                    ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  MODEL   : GPT-2 Small (117M params — tiny and fast)        ║
║  PYTHON  : 3.12 compatible ✅                                ║
║  GPU     : Free Colab T4                                     ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  PART A — SELF-SUPERVISED (no labels)                        ║
║  Data   : Raw TechNova HR document                           ║
║  Method : Next-word prediction at every token position       ║
║  Labels : Auto-generated from document itself ✅             ║
║  Result : Model learns domain vocabulary and facts           ║
║                                                              ║
║  PART B — SUPERVISED (with labels)                           ║
║  Data   : 10 hand-crafted Q&A pairs from TechNova doc        ║
║  Method : Question → Exact Answer pattern learning           ║
║  Labels : Explicit correct answers ✅                        ║
║  Result : Model learns precise Q→A response format           ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  KEY NUMBERS:                                                ║
║  GPT-2 hidden dim  : 768 (not 4096 — that is LLaMA)         ║
║  Wq matrix shape   : 768 × 768  =  589,824 parameters       ║
║  Training time     : 3-5 minutes on free T4 GPU              ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  🎓 Suresh D R | AI Product Developer & Technology Mentor    ║
╚══════════════════════════════════════════════════════════════╝
```